# M30 · Travel times in a layered crust

**Part 2 · week 3 · about 9.2 h core + 1 h stretch**

## 한국어 요약

이 모듈에서는 수평 성층 지각 모델(kim1983, kim2011)에서 지진파의 주시를 직접 계산합니다. 파선 변수를 조절해 원하는 진앙 거리에 도달하는 직접파를 찾는 슈팅 방법으로 직접파 주시 함수를 작성하고, 반무한 공간 공식, 페르마 원리에 따른 전수 탐색, ObsPy의 TauP와 비교해 검증합니다. 경계면을 따라 진행하는 선두파의 주시, 임계 거리와 교차 거리를 계산하고, 이 과정에서 진원 결정에 쓰는 관측의 범위를 생각해 봅니다. M20에서 직접 고른 2019년 상주 지진(ML 3.9)의 P파와 S파 도달 시각으로 와다티 도표를 그려 발생 시각과 Vp/Vs 비를 구하고 기상청 발생 시각과 비교합니다. 마지막으로 각 모델에서 가장 먼저 도착하는 파(최초 도달파)의 주시와 여러분의 피킹 사이의 잔차로 두 속도 모델을 비교해 공통 이동, 거리에 따른 경향, 흩어짐이 각각 무엇을 뜻하는지 읽고, 실무 도구인 TauP를 같은 조건으로 다시 실행한 뒤 변수 하나를 바꾸어 그 차이를 설명합니다.

## What you will be able to do

- Compute direct-wave travel times in a flat layered crust by shooting on the ray parameter, and test them against a homogeneous half-space, a brute-force search and TauP
- Compute head-wave times, critical distances and crossover distances for the Sangju source depth, and explain from them which arrivals a locator here should use
- Estimate the origin time and Vp/Vs of the Sangju earthquake from your own picks with a Wadati diagram, and compare them with the KMA catalog
- Compare two published velocity models through the residuals of your picks against each model's first arrival, and say what a common shift, a trend with distance and scatter each mean
- Run TauP as a production tool (an unchanged rerun, then one changed parameter) and explain how its times differ from your own

## Before you start

- Modules: M10, M20
- Reading: Bormann & Wendt (2012), the whole exercise, by hand, before this module; Shearer (2019), the chapter on ray theory and travel times; Stein & Wysession (2003), the section on refracted waves in layered media; Havskov, Bormann & Schweitzer (2012), the sections on travel times and the Wadati diagram
- Needs: stp
- Key terms (see the glossary): travel time (주시), velocity model (속도 구조 모델), direct wave (직접파), half-space (반무한 공간), ray parameter (파선 변수), Snell's law (스넬의 법칙), takeoff angle (이탈각), slowness (느림도(속도의 역수)), shooting method (슈팅 방법(사격법)), bisection (이분법), Fermat's principle (페르마의 원리), head wave (선두파), critical angle (임계각), critical distance (임계 거리), crossover distance (교차 거리), first arrival (최초 도달파 (초동)), intercept time (절편 시간), Moho (crust-mantle boundary; 모호면, 모호로비치치 불연속면), Pn and Sn (Pn파와 Sn파), reduced travel time (환산 주시), epicentral distance (진앙 거리), hypocentral distance (진원 거리), origin time (진원시, 발생 시각), arrival time (도달 시각), phase pick (위상 피킹, 도달 시각 읽기), Wadati diagram (와다티 도표), S-P time (S-P 시간), Vp/Vs ratio (Vp/Vs 비), least squares (최소제곱법), residual (잔차), RMS residual (RMS 잔차), TauP (TauP), Earth-flattening transformation (지구 평탄화 변환), widget (위젯)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M30.yaml](../../../.sgtl/criteria/M30.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M30.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
EVENT_FOLDER = "20190721020418"           # its folder in your STP download (the UTC origin time, as a name)
PICKS_FILE = "work/picks/M20_manual_picks.csv"   # your manual picks of M20, relative to the repository folder
MAX_DISTANCE_KM = 100.0                   # Parts 4-6 use direct P and S picks out to this epicentral distance
X_MAX_KM = 200.0                          # travel-time curves run from 0 to this distance
CHANGED_DEPTH_KM = 10.0                   # task 5: the one parameter you change in the TauP run (source depth)
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M30")

import contextlib
import io
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read
from obspy.geodetics import gps2dist_azimuth

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M30-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up models whose answers are known from how
they were built (`st.reroll("M30-T1")` draws new ones), on the two shipped models through a separate program
(ObsPy's TauP, which your own functions may not use), and, in task 3, on your own picks; there the check
tests loose physical limits only and never prints a value.

The order of the work is the one of every module from here on: first you write the core function and test it
on cases with a known answer (tasks 1-3), then you apply it to real data and compare with an independent
reference (tasks 3 and 4), and last you run the production tool, an unchanged rerun first, then with one
parameter changed (task 5).

Before this module, do exercise EX 11.1 of the *New Manual of Seismological Observatory Practice* (NMSOP-2,
Bormann & Wendt 2012) by hand: it locates earthquakes with circles and chords from S-P times and needs no
computer. This module explains where the travel times behind such circles come from.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M30.md`.

## Part 1 · Two layered models of the crust (about 20 min)

The **travel time** (glossary `travel_time`) of a seismic wave is the time it takes from the source to a
station. To compute it you need a **velocity model** (glossary `velocity_model`): the wave speeds as a function
of depth. The course ships two published one-dimensional models of the crust under South Korea: `kim1983` (Kim
& Kim 1983) and `kim2011` (Kim, Rhie & Kim 2011). Each row of the table is a flat layer: it starts at depth
`top_km` and reaches down to the top of the next row; the speeds `vp` and `vs` (km/s) are the same everywhere
inside it. The last row has no bottom: it is a **half-space** (glossary `half_space`), a layer that continues
downward forever. In both models the last row is the upper mantle, and its top is the **Moho** (glossary
`crust_mantle_boundary`), the boundary between the crust and the mantle.

The cell also reads the Sangju earthquake from the course's KMA catalog. Note how precise the catalog is: the
epicenter to 0.1 degree (about 10 km), the depth to the kilometre, the origin time to the second.

In [ ]:
KIM1983 = st.data.velocity_model("kim1983")
KIM2011 = st.data.velocity_model("kim2011")
for model in (KIM1983, KIM2011):
    print(f"{model.attrs['name']}: {model.attrs['citation']}")
    print(model.to_string(index=False), "\n")

catalog = st.data.kma_catalog()
event = catalog.loc[catalog["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN_KMA = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
DEPTH_KM = float(event["depth_km"])
print(f"KMA {KMA_EVENT_ID}: origin {ORIGIN_KMA} (UTC), epicenter {EV_LAT:.2f} N {EV_LON:.2f} E, depth "
      f"{DEPTH_KM:g} km, ML {event['ml']:.1f}")
print(f"source of the catalog: {catalog.attrs.get('source', 'KMA')}")

## Part 2 · Rays through flat layers (about 30 min, then task 1)

Think of the wave as many **rays** leaving the source, each a line along which the wave front travels. The
**direct wave** (glossary `direct_wave`) to a station is the ray that leaves the source upward and reaches the
station without being reflected on the way. Inside one layer a ray is straight. Where it crosses a boundary it
bends by **Snell's law** (glossary `snells_law`): sin(i) / v stays the same on both sides, where i is the angle
between the ray and the vertical and v the speed of the layer. The number that stays the same along the whole
ray is the **ray parameter** (glossary `ray_parameter`),

$$p = \frac{\sin i_j}{v_j}\quad\text{in every layer } j,$$

in s/km. 1/v is called the **slowness** (glossary `slowness`), so p is the horizontal part of the slowness.
The angle at which the ray leaves the source is its **takeoff angle** (glossary `takeoff_angle`).

A ray with ray parameter p that crosses a layer of thickness $h_j$ moves sideways by $h_j \tan i_j$ and takes
$h_j / (v_j \cos i_j)$ seconds. For a source at depth z the direct ray crosses only the layers above the source
(in the source layer, only the part above the source). Adding up gives the distance and the time of that ray:

$$X(p) = \sum_j h_j \tan i_j, \qquad T(p) = \sum_j \frac{h_j}{v_j \cos i_j}, \qquad \sin i_j = p\,v_j .$$

For p = 0 the ray goes straight up (X = 0). As p grows the ray tilts, and as p approaches 1 / (the largest
speed it crosses) the ray in that layer runs almost sideways and X grows without limit. So every distance x is
reached by exactly one p. The **shooting method** (glossary `shooting_method`) finds it by trial: fire a ray,
see where it lands, adjust p, fire again. **Bisection** (glossary `bisection`) does the adjusting: keep an
interval of p that contains the answer and halve it, again and again, keeping the half where X(p) passes x.
Every halving gains one binary digit, so about 50 halvings reach the precision of the computer.

Two checks of any travel-time code. In a single layer (a homogeneous half-space) the ray is a straight line and
$T = \sqrt{x^2 + z^2} / v$. And by **Fermat's principle** (glossary `fermats_principle`) the ray is the path
of least time: no other path between the source and the station is quicker. Your task 1 is checked both ways,
and also against TauP (Part 6), a program that seismologists use every day.

### M30-T1 · Direct-wave travel time by shooting · core · about 120 min

**Goal:** write `direct_time(x_km, depth_km, model, phase="P")`: the travel time of the direct wave in a flat
layered model, found by shooting on the ray parameter.

**Given:** the two formulas of Part 2 and numpy. Not allowed in this function: `obspy.taup`, `pyrocko` and the
course package (the check compares with TauP itself).

**Contract:**
- Inputs: `x_km`, the epicentral distance (km, from 0 up) as one number or a 1-D numpy array; `depth_km`, the
  source depth (km, > 0); `model`, a table like `KIM1983` (columns `top_km`, `vp`, `vs`; the first top is 0, the
  last row is a half-space; do not change the table); `phase`, `"P"` (use `vp`) or `"S"` (use `vs`).
- The station is at the surface. A source exactly on a boundary belongs to the layer below; the direct ray
  crosses only what lies above the source.
- Output: the travel time in seconds, a float for one distance and a 1-D numpy array of the same length for an
  array, within 5 ms of the true time out to 150 km.
- Example: in a single layer of 6 km/s, a source 10 km deep and a station 24 km away give 26 / 6 = 4.3333 s.

**Done when** `st.check("M30-T1", func=direct_time)` passes: on 6 one-layer models (the formula above), on 4
two-layer models (Fermat's principle by brute force: the time through every crossing point on the boundary),
and on `kim1983`, `kim2011` and a random four-layer model through TauP at 0-150 km.

**Stuck?** `st.hint("M30-T1", 1)`

In [ ]:
# Before you code: a source 14 km deep and a station 100 km away. Is the straight line from the source to the
# station the quickest path in kim1983? In kim2011? Will the direct P arrive earlier in kim1983 or in kim2011,
# and roughly by how much? Write down your guesses with a reason.
st.predict("M30-T1", """

""")

In [ ]:
def direct_time(x_km, depth_km, model, phase="P"):
    """Travel time (s) of the direct wave from a source at depth_km to a surface station x_km away.

    x_km     : epicentral distance(s) in km, one number or a 1-D numpy array (>= 0)
    depth_km : source depth in km (> 0); the direct ray crosses only the layers above it
    model    : table with top_km, vp, vs (km/s); the last row is a half-space; left unchanged
    phase    : "P" (vp) or "S" (vs)
    returns  : a float for one distance, a 1-D numpy array for an array of distances
    """
    # Step 1: the layers above the source: thickness h_j (the source layer only down to depth_km) and speed v_j.
    # Step 2: for one ray parameter p: sin(i_j) = p v_j; X(p) = sum h_j tan(i_j); T(p) = sum h_j / (v_j cos(i_j)).
    # Step 3: p runs from 0 to 1 / max(v_j). Bisection on p until X(p) = x (for every distance), then T(p).
    # YOUR CODE HERE
    raise NotImplementedError("M30-T1: write your code here")

In [ ]:
st.check("M30-T1", func=direct_time)

Your function on the example of the Contract, and on the two models for a source at the KMA depth:

In [ ]:
one_layer = pd.DataFrame({"top_km": [0.0], "vp": [6.0], "vs": [3.5]})
print(f"example of the Contract: {direct_time(24.0, 10.0, one_layer):.4f} s")
x_show = np.array([0.0, 10.0, 30.0, 50.0, 100.0, 150.0])
table = pd.DataFrame({"x_km": x_show})
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    for phase in ("P", "S"):
        table[f"{name} {phase} (s)"] = direct_time(x_show, DEPTH_KM, model, phase)
upper = KIM2011[KIM2011["top_km"] < DEPTH_KM]                    # kim2011's layers above the source (Part 1)
km_in_layer = np.diff(np.append(upper["top_km"].to_numpy(), DEPTH_KM))
mean_slowness = float(np.sum(km_in_layer / upper["vp"].to_numpy())) / DEPTH_KM    # P along a straight line (s/km)
table["straight line in kim2011, P (s)"] = np.hypot(x_show, DEPTH_KM) * mean_slowness
print(f"direct waves from a source at {DEPTH_KM:g} km:")
print(table.round(3).to_string(index=False))

**Explain (3-5 sentences):** Use the table above. Where is kim2011 faster than kim1983, and why (look at the
layer each source sits in)? How much longer than your direct time is the straight line through kim2011 at 100
km, and why can the bent ray be quicker although it is longer? Why can the direct wave not use the fast layers
below the source, however far the station?

## Part 3 · Head waves, critical and crossover distances (about 30 min, then task 2)

A ray that goes down from the source can meet a boundary with a faster layer below it. At one angle, the
**critical angle** (glossary `critical_angle`), sin(i) = v_above / v_below, the refracted ray runs along the
boundary in the faster layer. It sends energy back up at the same angle all along its way: this is the **head
wave** (glossary `head_wave`). Its ray parameter is p = 1 / v_below in every layer it crosses. The head wave
along the Moho is called **Pn** for P and **Sn** for S (glossary `pn_sn`).

The head wave starts at the **critical distance** (glossary `critical_distance`): it needs room to go down to
the boundary and come back up at the critical angle, so closer stations never see it. Beyond the critical
distance its time grows along a straight line, $t = x / v_{below} + t_i$, where the **intercept time**
(glossary `intercept_time`) $t_i$ holds the depth information. Because it runs fast along the boundary, the head
wave overtakes the direct wave at the **crossover distance** (glossary `crossover_distance`); beyond it the head
wave arrives first. The earliest wave at a station, whichever path it took, is the **first arrival** (glossary
`first_arrival`): the onset that a picker marks.

A head wave can only exist along a boundary below the source, and only if every layer above that boundary is
slower than the layer below it (a faster layer in between would reflect the ray before it reaches the critical
angle). The head wave goes down from the source (crossing the layers between the source and the boundary) and
comes up to the station (crossing every layer above the boundary). In each of these layers, a leg of thickness
$d$ crosses at the critical angle of that layer.

In this course head waves are a travel-time idea only: a lab rule says that earthquakes are located with
direct P and S waves within about 100 km, and never with Pn or Sn. Task 2 lets you see why.

### M30-T2 · Head waves along a boundary · core · about 90 min

**Goal:** write `head_wave(x_km, depth_km, model, phase="P", interface=-1)`: the head-wave times along the top of
one layer, its critical distance, and the distance where it overtakes your direct wave.

**Given:** Part 3, your `direct_time` of task 1 (use it for the crossover) and numpy. Not allowed:
`obspy.taup`, `pyrocko` and the course package.

**Contract:**
- Inputs: `x_km`, `depth_km`, `model` and `phase` as in task 1; `interface`, the row number (Python indexing) of
  the layer along whose top the head wave runs: `-1`, the last row, is the half-space below the Moho in both
  shipped models (Pn, Sn); `1` is the boundary at the top of the second row.
- Output: a tuple `(times, critical_km, crossover_km)`: `times` in s (a float for one distance, a 1-D numpy
  array for an array), NaN at distances before the critical distance; `critical_km`, the critical distance (km);
  `crossover_km` (km), beyond which the head wave arrives before your `direct_time` (to better than 0.1 km).
- When no head wave can run along that boundary (the source is not above it, or a layer above it is at least
  as fast as the layer below it), all three are NaN.
- Example: one layer of 6 km/s and 10 km over a half-space of 8 km/s, source 5 km deep, P: the critical angle
  has sin(i) = 0.75, the legs are 10 + 5 = 15 km, so the critical distance is 15 tan(i) = 17.01 km, and at
  x = 100 km the time is 100 / 8 + 15 cos(i) / 6 = 14.1536 s.

**Done when** `st.check("M30-T2", func=head_wave)` passes: on 8 cases (kim1983 at 14 km for both of its
boundaries, kim2011 with sources above and below a boundary, a random three-layer model, and a model with a
faster middle layer). Times within 5 ms and critical distances within 0.02 km of Fermat's principle applied leg
by leg; crossover distances within 1 % (+ 0.1 km) of where the direct wave (the half-space formula, or the
brute force of task 1) meets that head wave.

**Stuck?** `st.hint("M30-T2", 1)`

In [ ]:
# Before you code: for the Sangju source (14 km) in kim1983, beyond what distance do you expect the first P
# arrival to stop being the direct wave? Which boundary's head wave overtakes it first, the one at 15 km or the
# Moho at 32 km? Give a distance for each and your reasoning.
st.predict("M30-T2", """

""")

In [ ]:
def head_wave(x_km, depth_km, model, phase="P", interface=-1):
    """Head wave along the top of layer `interface`: (times, critical distance, crossover distance).

    x_km      : epicentral distance(s) in km, one number or a 1-D numpy array
    depth_km  : source depth in km
    model     : table with top_km, vp, vs (km/s); left unchanged
    phase     : "P" or "S"
    interface : row number of the layer below the boundary (-1: the half-space, the Moho in the shipped models)
    returns   : (times in s, NaN before the critical distance; critical distance in km; crossover distance in
                km, where the head wave overtakes your direct_time); all NaN when no head wave can exist
    """
    # Step 1: the boundary: n = the row of the layer below it, z_n its top, v_n its speed; ray parameter 1 / v_n.
    # Step 2: no head wave (all NaN) unless the source is above z_n and every layer above z_n is slower than v_n.
    # Step 3: legs: each layer above z_n is crossed once going up (all of it) and, below the source, once going
    #         down. With sin(i) = v_layer / v_n: critical distance = sum of legs * tan(i);
    #         time = x / v_n + sum of legs * cos(i) / v_layer, from the critical distance on.
    # Step 4: crossover: where your direct_time(x) equals the head-wave time; bisection on the sign of the
    #         difference, between the critical distance and a distance where the head wave is already earlier.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T2: write your code here")

In [ ]:
st.check("M30-T2", func=head_wave)

Your two functions for the Sangju source depth: every boundary below the source, in both models, for P and S.

In [ ]:
rows = []
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    for n in range(1, len(model)):
        for phase in ("P", "S"):
            t100, x_crit, x_cross = head_wave(100.0, DEPTH_KM, model, phase, n)
            rows.append({"model": name, "boundary (km)": model["top_km"].iloc[n], "phase": phase,
                         "critical (km)": x_crit, "crossover (km)": x_cross, "head wave at 100 km (s)": t100,
                         "direct at 100 km (s)": direct_time(100.0, DEPTH_KM, model, phase)})
crossovers = pd.DataFrame(rows)
print(f"source depth {DEPTH_KM:g} km (NaN: no head wave along that boundary for this source)")
print(crossovers.round(2).to_string(index=False))

x_grid = np.linspace(0.0, X_MAX_KM, 401)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=True)
for ax, (name, model) in zip(axes, (("kim1983", KIM1983), ("kim2011", KIM2011))):
    ax.plot(x_grid, direct_time(x_grid, DEPTH_KM, model, "P") - x_grid / 8.0, color="k", lw=1.6, label="Direct P")
    below = [n for n in range(1, len(model)) if model["top_km"].iloc[n] > DEPTH_KM]
    for k, (n, color) in enumerate(zip(below, ("tab:green", "tab:purple", "tab:orange"))):
        t_head, _, x_cross = head_wave(x_grid, DEPTH_KM, model, "P", n)
        if np.isfinite(t_head).any():
            ax.plot(x_grid, t_head - x_grid / 8.0, color=color, lw=1.3,
                    label=("First", "Second", "Third")[k] + " boundary below the source: head wave")
            ax.axvline(x_cross, color=color, lw=0.8, ls="--")
    ax.axvspan(MAX_DISTANCE_KM, X_MAX_KM, color="0.92", lw=0, zorder=0)
    tops_below = ", ".join(f"{model['top_km'].iloc[n]:g}" for n in below)
    ax.set_title(f"{name}: boundaries below the source at {tops_below} km", fontsize=10)
    ax.set_xlabel("Epicentral distance (km)")
axes[0].set_ylabel("Travel time - distance / 8 km/s (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "head_waves.png", dpi=150)

The figure shows **reduced travel time** (glossary `reduced_travel_time`): the time minus distance / 8 km/s.
Subtracting the same straight line from every curve tilts them so that small differences show; a wave moving
at 8 km/s would be flat. Dashed lines mark your crossover distances; the grey band lies beyond 100 km.

**Explain (3-5 sentences):** Compare the crossover distances in your table with your prediction. In kim1983,
which wave is the first P at 70-100 km, and why does that boundary overtake so early for this source? Where do
Pn and Sn become first arrivals in each model? Using these numbers, give two reasons why locators here keep to
direct P and S within about 100 km and never use Pn or Sn.

In [ ]:
# Optional explorer E4 (not checked): change the layers and the source depth; your own two functions draw.
st.widgets.traveltime_explorer("kim1983", direct=direct_time, head=head_wave, depth_km=DEPTH_KM)

## Part 4 · The Wadati diagram from your own picks (about 30 min, then task 3)

In M20 you picked the **arrival times** (glossary `arrival_time`) of P and S at stations around the Sangju
earthquake; each measured arrival is a **phase pick** (glossary `phase_pick`). The cell below reads your pick
file. If it reports no file, finish the picking task of M20 first.

Without any velocity model the picks already say two things. At one station the P wave arrives at
$t_P = t_0 + R / V_P$ and the S wave at $t_S = t_0 + R / V_S$, where $t_0$ is the **origin time** (glossary
`origin_time`) and R the **hypocentral distance** (glossary `hypocentral_distance`). Eliminating R gives

$$t_S - t_P = \left(\frac{V_P}{V_S} - 1\right)(t_P - t_0).$$

The **S-P time** (glossary `s_minus_p_time`) plotted against the P arrival time is therefore a straight line:
the **Wadati diagram** (glossary `wadati_diagram`). Its slope is Vp/Vs - 1 (glossary `vp_vs_ratio`), and it
reaches zero S-P time at the origin time. This holds for any path, as long as P and S share it and their speeds
keep the same ratio along the way. Fitting the line is a case of **least squares** (glossary `least_squares`):
choose the slope and the intercept that make the sum of squared differences between the points and the line
smallest. Havskov et al. (2012, NMSOP-2 IS 11.1) show the method.

In [ ]:
PICKS_PATH = st.data.repo_path(PICKS_FILE)
SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / EVENT_FOLDER
print("pick file:", PICKS_FILE, "| found" if PICKS_PATH.is_file() else "| NOT FOUND: run the picking task of M20")

In [ ]:
def read_picks(path, event_ids=(KMA_EVENT_ID, EVENT_FOLDER)):
    """Your pick file as a table: station (NET.STA), phase (P or S) and time (an obspy UTCDateTime, in UTC).

    It accepts the usual spellings of the columns (station or network + station or seed_id; phase; time_utc or
    time) and, when the file has an event column, keeps the rows of this earthquake.
    """
    raw = pd.read_csv(path, dtype=str).rename(columns=lambda c: c.strip().lower())
    when = next((c for c in ("time_utc", "time", "pick_time", "utc", "arrival_time", "arrival_utc") if c in raw),
                None)
    if when is None or "phase" not in raw:
        raise ValueError(f"{path}: need a phase column and a time column (time_utc), found {list(raw.columns)}")
    if "station" in raw:
        names = raw["station"].str.strip()
        if "network" in raw:
            names = names.where(names.str.contains(".", regex=False), raw["network"].str.strip() + "." + names)
    else:
        seed = raw["seed_id"] if "seed_id" in raw else raw["trace_id"]
        names = seed.str.split(".").str[:2].str.join(".")
    keep = pd.Series(True, index=raw.index)
    for col in ("event_id", "event", "utc_id", "event_folder"):
        if col in raw and raw[col].isin(event_ids).any():
            keep = raw[col].isin(event_ids)
    table = pd.DataFrame({"station": names, "phase": raw["phase"].str.strip().str.upper().str[0],
                          "time": [UTCDateTime(t) for t in raw[when].str.strip()]})[keep.to_numpy()]
    return table[table["phase"].isin(["P", "S"])].reset_index(drop=True)


picks = read_picks(PICKS_PATH) if PICKS_PATH.is_file() else None
if picks is not None:
    first = picks.sort_values("time").groupby(["station", "phase"]).first()["time"].unstack("phase")
    pairs = first.dropna()
    tp_s = np.array([t - ORIGIN_KMA for t in pairs["P"]])       # seconds after the KMA origin time
    ts_s = np.array([t - ORIGIN_KMA for t in pairs["S"]])
    print(f"{len(picks)} picks; {len(first)} stations, {len(pairs)} with both P and S")
    print(pd.DataFrame({"station": pairs.index, "P after KMA origin (s)": tp_s, "S after KMA origin (s)": ts_s,
                        "S-P (s)": ts_s - tp_s}).round(3).to_string(index=False))

### M30-T3 · The Wadati diagram · core · about 60 min

**Goal:** write `wadati_fit(tp_s, ts_s)`: the origin time and Vp/Vs from P and S arrival times at the same
stations, by a least-squares line through the Wadati diagram.

**Given:** Part 4 and numpy (means and sums). Not allowed in this function: `numpy.polyfit` (also as
`numpy.ma.polyfit`), `numpy.polynomial`, `numpy.linalg.lstsq`, all of `scipy`, `sklearn`, `statsmodels`,
`torch`, `xarray`, `obspy.signal` and the course package: you write the least-squares line yourself.

**Contract:**
- Inputs: `tp_s`, `ts_s`: 1-D arrays (or lists) of equal length (at least 2), the P and S arrival times at the
  same stations in the same order, in seconds after one reference moment of your choice.
- Fit the line y = k x + c by ordinary least squares with x = tp and y = ts - tp (the S-P time on the P time).
- Output: a tuple `(t0_s, vp_vs)` of floats: t0, where the line reaches y = 0 (in seconds after the same
  reference moment), and Vp/Vs = 1 + k.
- Example: tp = 10, 20, 30 s and ts = 17, 34, 51 s give (0.0, 1.7).

**Done when** `st.check("M30-T3", func=wadati_fit, picks=picks, event=KMA_EVENT_ID)` passes: on 12 made-up sets
of 5-15 stations (exact lines, and lines with scatter compared with a library fit), then on your own Sangju
picks, where it only tests loose physical limits (written in `criteria/M30.yaml`) and prints no value.

**Stuck?** `st.hint("M30-T3", 1)`

In [ ]:
# Before you code: what Vp/Vs do you expect in the upper crust under Sangju (compare the vp/vs of the two models
# in Part 1)? The KMA catalog gives the origin time to the whole second: will your Wadati origin time fall before
# or after 02:04:18, and by how much at most if the catalog were only rounded?
st.predict("M30-T3", """

""")

In [ ]:
def wadati_fit(tp_s, ts_s):
    """Origin time and Vp/Vs from a least-squares line through the Wadati diagram.

    tp_s, ts_s : P and S arrival times at the same stations (s after one reference moment), equal length >= 2
    returns    : (t0_s, vp_vs): where the S-P time on the line is zero (s after the same moment), and 1 + slope
    """
    # Step 1: x = tp, y = ts - tp. A straight line y = k x + c.
    # Step 2: least squares: k = sum((x - mean x)(y - mean y)) / sum((x - mean x)^2), c = mean y - k mean x.
    # Step 3: y = 0 at x = -c / k (the origin time); Vp/Vs = 1 + k.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T3: write your code here")

In [ ]:
st.check("M30-T3", func=wadati_fit, picks=picks, event=KMA_EVENT_ID)

In [ ]:
t0_s, vp_vs = wadati_fit(tp_s, ts_s)
ORIGIN_WADATI = ORIGIN_KMA + t0_s
spread = (ts_s - tp_s) - (vp_vs - 1.0) * (tp_s - t0_s)
print(f"your Wadati fit, {tp_s.size} stations: Vp/Vs = {vp_vs:.3f}, origin {ORIGIN_WADATI} "
      f"({t0_s:+.2f} s after the KMA origin time {ORIGIN_KMA})")
print(f"scatter of the points about your line: RMS {np.sqrt(np.mean(spread ** 2)):.3f} s, largest "
      f"{np.abs(spread).max():.3f} s")
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    print(f"{name} vp/vs of its layers:", np.round(model["vp"] / model["vs"], 3).tolist())

fig, ax = plt.subplots(figsize=(6.0, 4.2))
line_x = np.array([min(t0_s, 0.0) - 0.5, tp_s.max() + 1.0])
ax.plot(tp_s, ts_s - tp_s, "o", color="tab:blue", ms=5, label="Your picks")
ax.plot(line_x, (vp_vs - 1.0) * (line_x - t0_s), color="k", lw=1.2, label=f"Your line: Vp/Vs = {vp_vs:.3f}")
ax.axvline(0.0, color="tab:red", lw=1.0, ls="--", label="KMA origin time")
ax.plot([t0_s], [0.0], "*", color="gold", mec="k", ms=12, label=f"Your origin time ({t0_s:+.2f} s)")
ax.set_xlabel("P arrival time after the KMA origin time (s)")
ax.set_ylabel("S-P time (s)")
ax.set_title("Wadati diagram, Sangju 2019", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.86))
fig.savefig(Path(FIGURE_DIR) / "wadati_diagram.png", dpi=150)

**Explain (3-5 sentences):** Give your Vp/Vs and origin time (printed above) and compare them with your
prediction and with the vp/vs of the model layers. How far is your origin time from the KMA origin time, and can
the rounding to whole seconds explain it? Which points lie farthest from the line, and what could make a single
station fall off it? Why does the Wadati diagram need no velocity model and no location?

## Part 5 · Residuals: how well does a model fit? (about 30 min, then task 4)

A **residual** (glossary `residual`) is an observed travel time minus the time a model predicts; positive means
the wave came later than the model says. Your picks mark the first onset at each station, so the time to compare
them with is the model's first arrival at that distance: the earliest of its direct wave (task 1) and its head
waves along every boundary (task 2). Your table of Part 3 shows where a head wave overtakes the direct wave for
this source; within `MAX_DISTANCE_KM` neither Pn nor Sn does, so every first arrival here travels in the crust.

Four patterns tell different stories:

- **A common shift** (all residuals, P and S alike, off by the same amount): the origin time is off.
- **A trend with distance** (residuals growing or falling steadily with distance): the waves cross the stations
  at another speed than the model's first arrival. A negative slope means that they come ever earlier than the
  model predicts, so along those paths the real crust is faster than the model.
- **A split between P and S** (after one common shift, the P residuals on one side of zero and the S residuals
  on the other): the model's S-P times are too long or too short, so its Vp/Vs differs from that of the real
  paths (compare your Wadati value of task 3).
- **Scatter** (residuals jumping from station to station): picking errors, an error in the epicenter (the KMA
  epicenter is given to 0.1 degree, about 10 km; such an error shows as a pattern with **azimuth**, the
  direction to the station), and three-dimensional structure that no 1-D model has.

The **RMS residual** (glossary `rms_residual`), the root of the mean of the squared residuals, sums up a misfit
in one number. Husen & Hardebeck (2010) discuss what residuals can and cannot tell about a location.

The cell below builds the table of observed travel times: your picks within `MAX_DISTANCE_KM`, the epicentral
distance (glossary `epicentral_distance`) from the KMA epicenter to each station (station coordinates from the
SAC headers of your download), and the travel time counted from your Wadati origin time.

In [ ]:
def station_coordinates(names):
    """(latitude, longitude) of each NET.STA from the SAC headers of your download (any band)."""
    found = {}
    for path in sorted(SAC_DIR.glob("*/*.sac")):
        key = ".".join(path.name.split(".")[1:3])
        if key in names and key not in found:
            head = read(str(path), headonly=True)[0].stats.sac
            found[key] = (float(head.stla), float(head.stlo))
    return found


coords = station_coordinates(set(picks["station"]))
rows = []
for (sta, phase), group in picks.sort_values("time").groupby(["station", "phase"]):
    if sta in coords:
        meters, azimuth, _ = gps2dist_azimuth(EV_LAT, EV_LON, *coords[sta])
        rows.append({"station": sta, "phase": phase, "epi_km": meters / 1000.0, "azimuth": azimuth,
                     "tt_s": group["time"].iloc[0] - ORIGIN_WADATI})
obs = pd.DataFrame(rows)
obs = obs[obs["epi_km"] <= MAX_DISTANCE_KM].sort_values(["epi_km", "phase"]).reset_index(drop=True)
print(f"{len(obs)} travel times ({(obs['phase'] == 'P').sum()} P, {(obs['phase'] == 'S').sum()} S) at "
      f"{obs['station'].nunique()} stations within {MAX_DISTANCE_KM:g} km; counted from your Wadati origin time")
print(obs.round(3).to_string(index=False))

### M30-T4 · Travel-time residuals of two models · core · about 90 min

**Goal:** write `residual_summary(obs, depth_km, model, direct, head)`: the residuals of a table of observed
travel times against one model's first arrival, the one common shift, the RMS after it, and the trend with
distance of each phase.

**Given:** Part 5, your `direct_time` (task 1) and `head_wave` (task 2), and numpy. Not allowed: the same
libraries as in task 3 (write the least-squares slopes yourself; you may reuse your own code of task 3). The cell
below the task holds a colleague's comparison of the two models: run it as it is before your prediction.

**Contract:**
- Inputs: `obs`, a table with at least the columns `phase` (`"P"` or `"S"`), `epi_km` and `tt_s` (observed
  travel time, s), in any order and with any index (do not change it); `depth_km`; `model`; `direct`, a function
  called as `direct(x_km, depth_km, model, phase)` (like your `direct_time`); `head`, a function called as
  `head(x_km, depth_km, model, phase, interface)` that returns `(times, critical_km, crossover_km)` (like your
  `head_wave`). Use these two functions, with the depth and the model given.
- The predicted time of a row is the model's first arrival at that row's distance for that row's phase: the
  earliest of the direct wave and the head waves along every boundary of the model (`interface` from 1 up to the
  number of rows - 1). A head-wave time of NaN (no head wave at that distance) does not count.
- Output: a dict with
  - `"residual_s"`: a 1-D numpy array, observed minus predicted, one value per row of `obs`, in its row order;
  - `"shift_s"`: the one number that, subtracted from every residual (P and S alike), makes the sum of their
    squares smallest;
  - `"rms_s"`: the root mean square of the residuals after subtracting `shift_s`;
  - `"slope_P"`, `"slope_S"`: the least-squares slope of the residual against `epi_km` for that phase alone
    (s/km), NaN when that phase has fewer than 2 rows.

**Done when** `st.check("M30-T4", func=residual_summary)` passes: on 6 made-up tables (models of one to three
layers, the source in the top layer, distances out to 220 km) whose observed times were built from the check's
own direct-wave and head-wave functions, which it hands to yours, plus residuals the check chose (slopes compared
with a library fit).

**Stuck?** `st.hint("M30-T4", 1)`

In [ ]:
# A colleague compared the two models like this and wrote down the verdict. Run it as it is.
verdict = {}
late_kma = obs["tt_s"].to_numpy() + (ORIGIN_WADATI - ORIGIN_KMA)       # travel times from the KMA origin time
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    predicted = np.array([direct_time(x, DEPTH_KM, model, ph) for x, ph in zip(obs["epi_km"], obs["phase"])])
    verdict[name] = float(np.sqrt(np.mean((late_kma - predicted) ** 2)))
    print(f"colleague: {name} RMS residual {verdict[name]:.3f} s (KMA origin time)")
print("colleague's verdict: the model that fits Sangju better is", min(verdict, key=verdict.get))

In [ ]:
# Before you code: do you accept the colleague's verdict? What does a locator do with the origin time, and how
# would that change the comparison? With which wave did the colleague compare your farthest picks in kim1983, and
# which wave arrives first there (your table of Part 3)? Predict which model your residual_summary will favor, and
# whether the KMA or your Wadati origin time will matter for its RMS.
st.predict("M30-T4", """

""")

In [ ]:
def residual_summary(obs, depth_km, model, direct, head):
    """Residuals of observed travel times against a model's first arrival, with their common shift, RMS and trends.

    obs      : table with phase ("P"/"S"), epi_km and tt_s (observed travel time, s); left unchanged
    depth_km : source depth (km)
    model    : velocity model table (top_km, vp, vs)
    direct   : function direct(x_km, depth_km, model, phase) -> s, like your direct_time
    head     : function head(x_km, depth_km, model, phase, interface) -> (times, critical, crossover), like your
               head_wave
    returns  : {"residual_s": array in row order, "shift_s": float, "rms_s": float, "slope_P": float,
                "slope_S": float}
    """
    # Step 1: for the rows of each phase, the model's first arrival: the earliest of the direct wave and the head
    #         waves along every boundary (interface 1 to the last row); a NaN head-wave time does not count.
    # Step 2: residual = observed - first arrival, in the row order of obs.
    # Step 3: the shift that minimizes sum((residual - shift)^2) over all rows (set the derivative to zero).
    # Step 4: RMS of (residual - shift); for P rows and for S rows on their own, the least-squares slope of the
    #         residual against epi_km (as in task 3), NaN with fewer than 2 rows.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T4: write your code here")

In [ ]:
st.check("M30-T4", func=residual_summary)

Your function on your own Sangju picks, for both models: first from your Wadati origin time, then from the KMA
origin time, and last with a stand-in for `head_wave` that never finds a head wave, so that every pick is
compared with the direct wave alone (as in the colleague's cell).

In [ ]:
def no_head_wave(x_km, depth_km, model, phase="P", interface=-1):
    """Called like head_wave, but there is never a head wave: NaN times, NaN distances."""
    return np.full(np.shape(x_km), np.nan), np.nan, np.nan


obs_kma = obs.assign(tt_s=obs["tt_s"] + (ORIGIN_WADATI - ORIGIN_KMA))
is_p = (obs["phase"] == "P").to_numpy()
summary, head_first = {}, {}
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    out = residual_summary(obs, DEPTH_KM, model, direct_time, head_wave)
    from_kma = residual_summary(obs_kma, DEPTH_KM, model, direct_time, head_wave)
    alone = residual_summary(obs, DEPTH_KM, model, direct_time, no_head_wave)
    summary[name] = out
    head_first[name] = out["residual_s"] - alone["residual_s"] > 1e-6    # first arrival earlier than the direct
    left = out["residual_s"] - out["shift_s"]
    print(f"{name}: shift {out['shift_s']:+.3f} s, RMS after the shift {out['rms_s']:.3f} s, slope P "
          f"{out['slope_P'] * 100:+.3f} and S {out['slope_S'] * 100:+.3f} s per 100 km; mean after the shift: "
          f"P {left[is_p].mean():+.3f} s, S {left[~is_p].mean():+.3f} s")
    print(f"   from the KMA origin time: shift {from_kma['shift_s']:+.3f} s, RMS after the shift "
          f"{from_kma['rms_s']:.3f} s")
    print(f"   a head wave is the first arrival at {head_first[name][is_p].sum()} of {is_p.sum()} P and "
          f"{head_first[name][~is_p].sum()} of {(~is_p).sum()} S picks; against the direct wave alone: RMS "
          f"{alone['rms_s']:.3f} s, slope P {alone['slope_P'] * 100:+.3f} and S {alone['slope_S'] * 100:+.3f} s "
          "per 100 km")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, name in zip(axes, ("kim1983", "kim2011")):
    out = summary[name]
    for phase, color in (("P", "tab:blue"), ("S", "tab:red")):
        rows_ph = (obs["phase"] == phase).to_numpy()
        x_ph = obs["epi_km"].to_numpy()[rows_ph]
        r_ph = out["residual_s"][rows_ph] - out["shift_s"]
        by_head = head_first[name][rows_ph]
        ax.plot(x_ph[~by_head], r_ph[~by_head], "o", color=color, ms=4, label=f"{phase}: direct wave first")
        ax.plot(x_ph[by_head], r_ph[by_head], "o", mfc="none", color=color, ms=5, label=f"{phase}: head wave first")
        if np.isfinite(out["slope_" + phase]):
            line = np.array([x_ph.min(), x_ph.max()])
            ax.plot(line, r_ph.mean() + out["slope_" + phase] * (line - x_ph.mean()), color=color, lw=1.0)
    ax.axhline(0.0, color="0.5", lw=0.8)
    ax.set_title(f"{name}: RMS {out['rms_s']:.3f} s after the shift", fontsize=10)
    ax.set_xlabel("Epicentral distance (km)")
axes[0].set_ylabel("Residual minus common shift (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "residuals_two_models.png", dpi=150)

**Explain (3-5 sentences):** Which model fits your Sangju picks better, by how much (RMS printed above), and does
it agree with the colleague's verdict? Why do the shift and the RMS behave as they do when you start from the
KMA instead of your Wadati origin time? In kim1983, at how many picks is a head wave the first arrival, and what
does comparing every pick with the direct wave alone do to the trend of the S residuals? After the shift, on
which side of zero do kim1983's P and S residuals sit, and what does that say about its Vp/Vs next to yours?

A locator estimates the origin time together with the hypocenter (module M31), so a common shift of all
residuals is absorbed there and is not a misfit of the model. What is left after the shift (a trend with
distance, a split between P and S, scatter) is what speaks about the model.

## Part 6 · TauP, a production tool (about 20 min, then task 5)

**TauP** (glossary `taup`; Crotwell et al. 1999) computes travel times and ray paths in a layered spherical
Earth; ObsPy contains a Python version (`obspy.taup`). It needs a model file in its own format (`.nd`): one
line per depth with depth, vp, vs and density, two lines at each boundary (the speeds just above and just below),
the word `mantle` at the Moho, and the whole Earth down to its center. The helper below writes such a file from
a shipped model: the layers, the half-space down to 200 km, then the deep Earth of the global model ak135 that
comes with ObsPy (deep layers do not change the direct waves at our distances). Three ObsPy calls then do the
rest:

```python
from obspy.taup.taup_create import build_taup_model    # turns the .nd file into a TauP model file (.npz)
from obspy.taup import TauPyModel                       # loads it: TauPyModel(model="<folder>/<name>.npz")
taup.get_travel_times(source_depth_in_km=..., distance_in_degree=..., phase_list=[...])  # a list of arrivals
```

Each arrival has a `.name` and a `.time` (s). TauP names phases by their path: in TauP `P` and `S` leave the
source downward, `p` and `s` leave it upward (see the ObsPy documentation of `obspy.taup`).

In [ ]:
def write_taup_nd(model, path, join_km=200.0):
    """Write a shipped layered model as a TauP '.nd' file: the layers (constant speeds), 'mantle' at the top of
    the last row, the half-space down to join_km, then the deep Earth of ak135 below join_km."""
    import obspy.taup

    tops = model["top_km"].to_numpy(float)
    bottoms = list(tops[1:]) + [join_km]
    lines = []
    for j, (top, vp, vs) in enumerate(zip(tops, model["vp"], model["vs"])):
        if j == len(tops) - 1:
            lines.append("mantle")
        lines += [f"{top:.3f} {vp:.4f} {vs:.4f} 2.70", f"{bottoms[j]:.3f} {vp:.4f} {vs:.4f} 2.70"]
    ak135 = Path(obspy.taup.__file__).parent / "data" / "ak135f_no_mud.nd"
    for row in ak135.read_text().splitlines():
        parts = row.split()
        if parts in (["outer-core"], ["inner-core"]):
            lines.append(parts[0])
        elif len(parts) >= 4 and float(parts[0]) > join_km:
            lines.append(" ".join(parts[:4]))
    Path(path).write_text("\n".join(lines) + "\n")
    return Path(path)


print(write_taup_nd(KIM1983, Path(RESULT_DIR) / "kim1983.nd").read_text().splitlines()[:9])

### M30-T5 · Travel times from TauP · core · about 60 min

**Goal:** write `taup_times(model_name, depth_km, x_km)`: the direct P and S travel times that TauP computes for a
shipped model.

**Given:** the helper `write_taup_nd` and the three ObsPy calls of Part 6.

**Contract:**
- Inputs: `model_name`, a shipped model (`"kim1983"` or `"kim2011"`); `depth_km`, the source depth (km); `x_km`, a
  1-D numpy array of epicentral distances (km).
- Build the TauP model from the shipped layers (helper, then `build_taup_model` into a folder of your own,
  then `TauPyModel`). TauP wants the distance in degrees: use an Earth radius of 6371 km.
- Output: a tuple `(tp, ts)` of 1-D numpy arrays, one time per distance: the earliest arrival of the direct
  waves `p` and `s`, NaN where TauP finds none. Two calls with the same input must give identical numbers.

**Done when** `st.check("M30-T5", func=taup_times)` passes: on kim1983 at 14 km (3-150 km) and on both models at
random depths, compared with TauP run by the check on the same layers, within 2 ms; and a second call of the
first case must repeat the first exactly.

**Stuck?** `st.hint("M30-T5", 1)`

In [ ]:
# Before you code: will a second run of TauP give exactly the same numbers? Compared with your direct_time for
# kim1983 at 14 km, will TauP's direct P be earlier, later or the same, and by how much at 100 km? What will
# moving the source from 14 km to CHANGED_DEPTH_KM do to the times at the nearest and at the farthest station?
st.predict("M30-T5", """

""")

In [ ]:
def taup_times(model_name, depth_km, x_km):
    """Direct P and S travel times (s) from TauP for a shipped layered model.

    model_name : "kim1983" or "kim2011"
    depth_km   : source depth (km)
    x_km       : 1-D numpy array of epicentral distances (km)
    returns    : (tp, ts): earliest 'p' and 's' arrival at every distance, NaN where TauP finds none
    """
    from obspy.taup import TauPyModel
    from obspy.taup.taup_create import build_taup_model

    # Step 1: write the model file with write_taup_nd into a new folder and build it there (build_taup_model).
    # Step 2: load the built .npz with TauPyModel.
    # Step 3: for every distance: degrees = km / (6371 pi / 180); the earliest 'p' and the earliest 's' arrival.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T5: write your code here")

In [ ]:
st.check("M30-T5", func=taup_times)

The production run: TauP at your stations, an unchanged rerun, then one changed parameter (the source depth),
and the difference from your own flat-layer function.

In [ ]:
stations = obs.drop_duplicates("station").sort_values("epi_km")
x_st = stations["epi_km"].to_numpy()
base = taup_times("kim1983", DEPTH_KM, x_st)
rerun = taup_times("kim1983", DEPTH_KM, x_st)
changed = taup_times("kim1983", CHANGED_DEPTH_KM, x_st)
same = all(np.array_equal(a, b, equal_nan=True) for a, b in zip(base, rerun))
print(f"unchanged rerun identical to the first run: {same}")
runs = pd.DataFrame({"station": stations["station"].to_numpy(), "epi_km": x_st,
                     "TauP P (s)": base[0],
                     "TauP - yours, P (s)": base[0] - direct_time(x_st, DEPTH_KM, KIM1983, "P"),
                     "TauP - yours, S (s)": base[1] - direct_time(x_st, DEPTH_KM, KIM1983, "S"),
                     f"P at {CHANGED_DEPTH_KM:g} km - at {DEPTH_KM:g} km (s)": changed[0] - base[0],
                     f"S at {CHANGED_DEPTH_KM:g} km - at {DEPTH_KM:g} km (s)": changed[1] - base[1]})
print(runs.round(4).to_string(index=False))
runs.to_csv(Path(RESULT_DIR) / "M30_taup_runs.csv", index=False)

x_line = np.linspace(1.0, 150.0, 60)
fig, ax = plt.subplots(figsize=(6.0, 3.8))
for depth, style in ((5.0, ":"), (DEPTH_KM, "-"), (25.0, "--")):
    tp_line, _ = taup_times("kim1983", depth, x_line)
    ax.plot(x_line, tp_line - direct_time(x_line, depth, KIM1983, "P"), color="k", ls=style, lw=1.2,
            label=f"Source at {depth:g} km")
ax.axhline(0.0, color="0.6", lw=0.8)
ax.set_xlabel("Epicentral distance (km)")
ax.set_ylabel("TauP P - your P (s)")
ax.set_title("kim1983: TauP against your flat-layer times", fontsize=10)
fig.legend(loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "taup_minus_flat.png", dpi=150)

**Explain (3-5 sentences):** Was the rerun identical? How large is TauP minus your time at your nearest and
farthest station, with which sign, and how does it change with distance and depth (figure)? TauP and your
function use the same layers: what is different (glossary `earth_flattening`; Chapman 1973), and does the
difference matter next to your picking uncertainty? What did the shallower source change at near and at far
stations, and what does that tell you about which stations constrain the depth of an earthquake?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M30")

### Going further (optional, not checked)

1. In the explorer E4, move the source of kim1983 from 14 km to 16 km (just below the boundary at 15 km). What
   happens to the head wave along 15 km, and to the first arrival at 70-100 km? Explain with Part 3.
2. Repeat task 4 with the source depth at 8 km and at 20 km. Which depth fits your picks best in each model?
   Can you tell the depth apart from the origin time with stations beyond 30 km only?
3. Draw the residuals of task 4 against the azimuth of each station. Is there a pattern that a shift of the
   epicenter by a few kilometres would remove? (Module M31 does exactly this.)
4. If you fetched the Suncheon dataset (about 23 km deep), redo the Wadati diagram for one of its earthquakes.

Next: M31 locates the Sangju earthquake with your direct P and S picks and your travel-time function.